# Lesson 13 Lab — NF4 and QLoRA: A 4-Bit Fine-Tuning Memory Ledger

**Puzzle:** If the frozen base model is four-bit, where does fine-tuning memory still go?

This notebook keeps the RTX 5090 outputs from a complete run. Read the theory cells, make a prediction, and then use **Run All** on your own GPU.


## Why this matters

QLoRA makes the base model cheap enough to keep frozen, but it does not make fine-tuning free. Activations, adapter parameters, gradients, optimizer states, temporary dequantization, and sequence length remain on the memory ledger. A useful feasibility calculation names each object instead of multiplying parameter count by four bits and stopping.


## 0. Predict before running

1. Estimate BF16 and ideal INT4 storage for seven billion parameters before opening the result.
2. Identify which tensors require gradients in a LoRA update and which remain frozen.
3. Explain why activation checkpointing can matter even when the base weights are four-bit.

For each answer, name the observation that would prove you wrong.


## 1. Name the concrete objects

QLoRA freezes a four-bit base, computes through a wider dtype, and trains LoRA matrices. The memory ledger still includes adapters, gradients, optimizer states, activations, temporary dequantization, and allocator reserve.

- QLoRA freezes a quantized base and trains small low-rank adapters.
- Optimizer state and gradients apply to trainable adapters, while activations remain a major runtime cost.
- NF4 is a non-uniform codebook designed for normally distributed weights.


## 2. Derive the mechanism

A rank-`r` adapter adds `ΔW = A·B` with roughly `r(in+out)` trainable parameters instead of `in×out`. NF4 provides a non-uniform 16-value codebook suited to normally distributed pretrained weights; double quantization compresses scale metadata.

A LoRA update writes `ΔW = BA`, where A and B have rank r much smaller than the full matrix dimensions. QLoRA keeps W frozen in a quantized representation, dequantizes as needed for compute, and backpropagates only into A and B. NF4 uses a non-uniform codebook designed for roughly normal weight distributions; double quantization compresses scale metadata, while paged optimizers address memory spikes.

The ledger separates persistent storage from training-time liveness. Ideal base bytes are `P·4/8`, but adapter weights, adapter gradients, two Adam moments, activations, and workspaces have their own dtype and multiplicity. Sequence length can dominate because saved activations scale with tokens even though base storage does not.

### Mechanism at a glance

```mermaid
flowchart LR
  N["NF4 base weights<br/>frozen"] --> D["blockwise dequantize"]
  D --> B["base linear output"]
  X["input activation"] --> B
  X --> L["trainable LoRA path"]
  B --> Y["combined output"]
  L --> Y
  Y --> G["gradients only for adapters"]
```

### Walk it step by step

1. **Freeze the quantized base.** The NF4 base weights are storage for forward computation, not trainable optimizer parameters.
2. **Dequantize for compute.** Blocks are reconstructed into the configured compute dtype as the layer executes.
3. **Train only adapters.** LoRA matrices, their gradients, and their optimizer states form the main trainable parameter budget.
4. **Keep a complete memory ledger.** Add quantized weights, scales, adapters, gradients, optimizer state, activations, and temporary workspace.


## 3. Verify the execution environment

The next cell asserts CUDA availability, fixes the seed, locates the lesson, and prints a sanitized GPU/PyTorch/CUDA record. Check it before interpreting output.


In [1]:
from pathlib import Path
import json
import sys
import torch

chapter_rel = Path("chapters/01-mixed-precision-int4")
repo_root = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / chapter_rel / "support" / "lab_common.py").exists()
)
sys.path.insert(0, str(repo_root / chapter_rel / "support"))
from lab_common import (base_result, cuda_benchmark, environment_record,
                        error_metrics, require_cuda, save_result,
                        symmetric_quantize)

lesson_dir = repo_root / chapter_rel / "13-nf4-qlora"
device = require_cuda()
torch.manual_seed(2026 + 13)
environment = environment_record()
print(json.dumps(environment, indent=2))


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "gpu_memory_gib": 31.358,
  "python": "3.12.13",
  "torch": "2.12.0",
  "cuda_runtime": "13.0"
}


## 4. Freeze the comparison

| Role | This run |
|---|---|
| Baseline | 7B BF16 base-weight arithmetic plus a frozen CUDA reference layer |
| Candidate | ideal INT4 base ledger with trainable low-rank adapters |
| Held constant | parameter count, adapter rank assumption, optimizer-state rule, toy layer shape |
| Measurements | base GiB, LoRA/Adam MiB, gradient finiteness, frozen-base flag, toy loss |
| Evidence | `pytorch-gpu` |

**Experiment:** Build a 7B-class memory ledger and run a CUDA low-rank adapter forward/backward over a frozen fake-quantized base matrix.


## 5. Read the experiment code

The lab combines a 7B-class arithmetic ledger with a real CUDA backward pass where only low-rank adapter tensors receive gradients.

The notebook first computes a transparent 7B ledger. It then runs a small forward/backward pass in which the fake-quantized base matrix has `requires_grad=False` and only low-rank adapter matrices receive gradients. The finite-gradient check proves the intended training path exists on CUDA.

The fake quantizer explains memory ownership but is not bitsandbytes NF4. The ledger also excludes full-model activations because they depend on architecture, microbatch, sequence length, checkpointing, and attention implementation.

Only after these variables match the protocol should the cell be executed.


In [2]:
params=7_000_000_000; rank=16; hidden=4096; layers=32
ledger={"bf16_base_gib":round(params*2/2**30,3),"int4_base_ideal_gib":round(params*0.5/2**30,3),
        "lora_trainable_mib":round(layers*2*hidden*rank*2/2**20,3),"adam_states_mib":round(layers*2*hidden*rank*8/2**20,3)}
dim=1024; base=torch.randn(dim,dim,device=device); _,_,base_q=symmetric_quantize(base,bits=4,group_size=128); base_q=base_q.detach()
a=torch.nn.Parameter(torch.randn(dim,rank,device=device)*0.01); b=torch.nn.Parameter(torch.zeros(rank,dim,device=device))
x=torch.randn(64,dim,device=device); target=torch.randn(64,dim,device=device)
loss=torch.nn.functional.mse_loss(x@base_q.t()+(x@a)@b,target); loss.backward()
result=base_result(13,"pytorch-gpu"); result.update({"seven_b_ledger":ledger,"toy_loss":round(loss.item(),7),
    "base_requires_grad":base_q.requires_grad,"adapter_grad_finite":bool(torch.isfinite(a.grad).all() and torch.isfinite(b.grad).all()),
    "conclusion":"The frozen four-bit base reduced weight storage, while adapters, optimizer state, and activations remained separate costs."})


## 6. Read the retained RTX 5090 result

**Recorded environment:** NVIDIA GeForce RTX 5090; compute capability 12.0; PyTorch 2.12.0; CUDA runtime 13.0.

| Measured field | Checked-in value |
|---|---:|
| 7B BF16 base | 13.039 GiB |
| 7B ideal INT4 base | 3.260 GiB |
| LoRA trainable state | 8.000 MiB |
| Adam states | 32.000 MiB |
| Base frozen | yes |
| Adapter gradients finite | yes |


## 7. Interpret rather than merely print

The arithmetic ledger placed a 7B BF16 base at 13.039 GiB and ideal four-bit storage at 3.260 GiB. Under the toy adapter assumptions, trainable LoRA weights occupied 8 MiB and two Adam moments 32 MiB. The base stayed frozen and adapter gradients were finite.

Those small adapter lines explain QLoRA's appeal, but the missing activation line can still be larger than the trainable state for long contexts. The result proves the ownership pattern and a toy CUDA backward pass, not a 7B end-to-end fine-tuning capacity number.

**Inspection rule:** Separate frozen base storage, trainable parameters, gradients, optimizer estimate, and activations.


## 8. Keep the evidence label honest

This run is labeled **`pytorch-gpu`**. The measured tensors and operations ran on CUDA through PyTorch. The result does not name a separate production backend unless an operator trace identifies it.

The next cell writes the complete structured result; its existing saved output is part of the checked-in evidence.


In [3]:
artifact_path = save_result(result, lesson_dir)
print(json.dumps(result, indent=2, sort_keys=True))
print("Saved: artifacts/rtx5090-result.json")


{
  "adapter_grad_finite": true,
  "base_requires_grad": false,
  "conclusion": "The frozen four-bit base reduced weight storage, while adapters, optimizer state, and activations remained separate costs.",
  "environment": {
    "compute_capability": "12.0",
    "cuda_runtime": "13.0",
    "gpu": "NVIDIA GeForce RTX 5090",
    "gpu_memory_gib": 31.358,
    "python": "3.12.13",
    "torch": "2.12.0"
  },
  "evidence_label": "pytorch-gpu",
  "executed_at_utc": "2026-08-07T14:45:40+00:00",
  "lesson": 13,
  "schema_version": 1,
  "seven_b_ledger": {
    "adam_states_mib": 32.0,
    "bf16_base_gib": 13.039,
    "int4_base_ideal_gib": 3.26,
    "lora_trainable_mib": 8.0
  },
  "toy_loss": 1036.6418457
}
Saved: artifacts/rtx5090-result.json


## 9. Make the bounded decision

> Four-bit base weights reduce one ledger line; sequence activations and adapter training state still control feasibility.

**Acceptance/rollback:** Reconcile theoretical and measured peak memory, confirm the base has no gradients, list compute dtype and optimizer, and validate downstream quality against a frozen baseline.

**Failure analysis:** Calling the base 'four-bit' while materializing a full BF16 copy defeats the ledger. Counting optimizer state for frozen weights overestimates memory, while omitting adapter moments underestimates it. A memory fit based on parameters alone can OOM during backward when saved activations and temporary buffers peak.


## 10. Extend the evidence

Run a real QLoRA step with bitsandbytes or another supported backend and measure `max_memory_allocated` by sequence length, microbatch, rank, and checkpointing policy. Compare predicted persistent bytes with observed peak, and explain the residual using allocator snapshots and activation liveness.

The full derivation, reproduction command, evidence boundary and primary references are in [`README.md`](README.md).
